# §56 — three arms on the §55 DoTA-train corpus: **K5dfieDoTA, K5dfibk25DoTA, K5dfiebk25DoTA**, s2024

Rule: `core/docs/final/KIM_RUN.md` **§56**, written before any number. Upload branch `v_final_las_vad` to `Thesis-Final/kat-vad` first. **Needs §55 done** (`k5dfidota.ipynb`: features, corpus, bakes, KNN and K5dfiDoTA on Drive). PRELIMINARY (n = 1). **Decides nothing; K5f stays the reference. Every row is in-domain on DoTA.**

| step | what | GPU | estimate |
|---|---|---|---|
| 1-4 | setup, deps, helpers, preflight (every §55 artifact must exist) | no | 3-5 min |
| 5 | tests — **raises if red** | no | 1-2 min |
| 6 | stage the §55 corpus, bake and KNN; check manifests | no | 5-10 min |
| 7 | train the three arms (1,764 steps each) + guards | **yes** | 3 x K5dfiDoTA's time |
| 8 | reads per arm: DoTA-CAP-dev, DADA, TAD, DAD, Nexar val | yes | 40-60 min per arm |
| 9 | table vs K5dfiDoTA + DAD placebo -> **paste `REPORTS/k5dfidota_arms_report.md` to Claude** | no | 3 min |

A re-run skips what is on Drive (each arm is skipped once `verified.json` matches). If the session dies, re-run from the top: finished arms are not retrained. Thesis / Thesis-V2 are only read.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import json
import os
import pickle
import shutil
import subprocess
import sys
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read-only: data, cache/clip
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # read-only: Nexar_window, Nexar caches, census
DRIVE_F  = '/content/drive/MyDrive/Thesis-Final'  # code, §55 artifacts, K5dfiDoTA; every output of this notebook
os.environ['PROJECT_ROOT']       = DRIVE_F
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE_F}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_F}/outputs'

os.environ['REPO'] = f'{DRIVE_F}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v_final_las_vad there'
assert '## 56. Three arms on the §55' in (REPO / 'core/docs/final/KIM_RUN.md').read_text(), 'KIM_RUN.md has no §56 -- re-upload kat-vad/'
assert '--control' in (REPO / 'core/tools/dad_position_diag.py').read_text(), 'dad_position_diag has no --row / --control -- re-upload kat-vad/'
assert 'if not is_abnormal:\n            return NORMAL_CLASS' in (REPO / 'core/data/dataset.py').read_text(), \
    'core/data/dataset.py has no §31 fix -- re-upload kat-vad/'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'

K5       = constants.KIM_MAIN_ARM                  # "K5": names run / eval sub-folders only
CTRL     = constants.KIM_DOTA_TRAIN_ROW            # K5dfiDoTA: the control (§55, reused)
NEXROW   = constants.KIM_DVS_IGNORE_ROW            # K5dfi (Nexar-trained): printed row
LAG      = constants.ZS_LAGOVAD_ARM
ARMS     = constants.KIM_DOTA_TRAIN_ARMS           # {row: extra --set values}
IGN      = constants.DVS_ANCHOR_MODE_IGNORE
SEED     = constants.KIM_SEED
ENC      = constants.V5_ENCODER                    # V2-S
OFF      = constants.V2_OFF
DATASET  = constants.NEXAR_DATASET
T2NAME   = constants.DADA_ORIGIN_DATASET
CAP_DEV  = constants.V2_SPLIT_DOTA_CAP_DEV
NEXAR_SPLIT = constants.V2_SPLIT_NEXAR_VAL         # nexar_test is never read here
TAD_NAME = constants.TAD_DATASET
DAD_NAME, DAD_STRIDE, DAD_SET = constants.DAD_DATASET, constants.DAD_STRIDE, constants.DAD_OUTPUT_SET
assert len(ARMS) == 3 and CTRL not in ARMS, ARMS

V2_CACHE, V2_OUT = Path(DRIVE_V2) / 'cache', Path(DRIVE_V2) / 'outputs'
WINDOW   = Path(DRIVE_V2) / 'data' / f'{DATASET}_{constants.NEXAR_CONSTRUCTION_WINDOW}'   # defs of every Nexar read
ROOT     = constants.OUTPUT_ROOT / constants.KIM_DOTA_TRAIN_OUTPUT_DIRNAME
REPORTS  = ROOT / 'REPORTS'
BUILD    = REPORTS / constants.DOTA_TRAIN_BUILD_REPORT
NEX_EV   = constants.OUTPUT_ROOT / constants.KIM_DVS_IGNORE_OUTPUT_DIRNAME / NEXROW / 'eval'
LAG_DAD  = constants.OUTPUT_ROOT / constants.ZS_LAGOVAD_OUTPUT_DIRNAME / 'eval' / DAD_SET / LAG / constants.ZS_LAGOVAD_RUN
NEX, DST_SUF = constants.NEXAR_STATS_BAKE_SUFFIX, constants.DOTA_TRAIN_STATS_BAKE_SUFFIX
ZS_SRC_NAMES = {'dada': f'{T2NAME}_test_sources_nexar', 'dota': f'{CAP_DEV}_s3_from_s1_nexar',
                'tad': f'{TAD_NAME}_s8_nexar', DAD_SET: f'{DAD_NAME}_s{DAD_STRIDE}_nexar'}   # = §55's ZS_SRC
assert all(n.endswith(NEX) for n in ZS_SRC_NAMES.values())
ZS_DST   = {s: constants.CACHE_ROOT / 'v2' / 'A2_S' / (n[: -len(NEX)] + DST_SUF) for s, n in ZS_SRC_NAMES.items()}
T2       = constants.DATA_ROOT / T2NAME
T2_CENSUS = T2 / 'counts' / 'resolved_census.json'
ANNOT    = constants.DATA_ROOT / 'DADA2000Origin' / constants.DADA_ORIGIN_ROOT_DIRNAME / constants.DADA_ORIGIN_ANNOTATION_FILENAME
DOTA     = constants.DATA_ROOT / 'DoTA'
DOTA_S1  = Path(DRIVE) / 'cache' / 'clip' / 'DoTA_s1_ncc'
TAD      = constants.DATA_ROOT / TAD_NAME
DAD_DATA = Path(DRIVE_F) / 'data' / f'{DAD_NAME}_s{DAD_STRIDE}'
CENSUS   = V2_OUT / 'v2' / 'REPORTS' / 'nexar_n0' / 'census.json'
NEXAR_X  = V2_CACHE / 'clip' / constants.NEXAR_CLIP_S1_DATASET
NEXAR_U  = V2_CACHE / 'video' / ENC / f'{constants.NEXAR_VIDEO_S1_DATASET}_squash'
DATA     = Path(DRIVE_F) / 'data' / constants.DOTA_TRAIN_CORPUS_DIRNAME
BAKE     = constants.CACHE_ROOT / 'v2' / 'A2_S' / constants.DOTA_TRAIN_BAKE_DIRNAME
KNN      = constants.CACHE_ROOT / 'knn' / DATA.name / constants.KNN_CACHE_FILENAME
LOCAL    = Path('/content/k5dfidota_arms')
LOCAL.mkdir(parents=True, exist_ok=True)


def row_paths(row):
    run_dir = ROOT / row / K5 / f's{SEED}'
    return run_dir, run_dir / 'checkpoint_last.pt', ROOT / row / 'eval'


CTRL_RUN, _, CTRL_EV = row_paths(CTRL)
INPUTS = (('§55 build report', BUILD), ('§55 corpus windows', DATA / constants.WINDOWS_FILENAME),
          ('§55 DoTA-train bake', BAKE / constants.V2_INPUT_MANIFEST_FILENAME), ('§55 KNN', KNN),
          *[(f'§55 restated {s} bake', d / constants.V2_INPUT_MANIFEST_FILENAME) for s, d in ZS_DST.items()],
          ('K5dfiDoTA checkpoint', CTRL_RUN / 'checkpoint_last.pt'), ('K5dfiDoTA verified', CTRL_RUN / 'verified.json'),
          ('K5dfiDoTA config', CTRL_RUN / 'config.yaml'), ('K5dfiDoTA metrics', CTRL_RUN / 'metrics.jsonl'),
          ('K5dfiDoTA nexar_val read', CTRL_EV / 'nexar' / CTRL),
          *[(f'{r} {s} read', ev / s / K5) for r, ev in ((CTRL, CTRL_EV), (NEXROW, NEX_EV)) for s in ('dota', 'dada', 'tad', DAD_SET)],
          ('LaGoVAD DAD scores', LAG_DAD / 'scores'), ('Nexar_window defs', WINDOW / constants.DEFS_FILENAME),
          ('T2 meta', T2 / constants.META_FILENAME), ('T2 census', T2_CENSUS), ('DADA annotation', ANNOT), ('DoTA s1', DOTA_S1),
          ('DoTA metadata', DOTA / 'metadata_val.json'), ('TAD labels', TAD / constants.FRAME_LABELS_TEST_FILENAME),
          ('DAD labels', DAD_DATA / constants.FRAME_LABELS_TEST_FILENAME),
          ('Nexar census', CENSUS), ('Nexar CLIP s1', NEXAR_X), ('Nexar V2-S s1', NEXAR_U))
missing = [f'{n}: {p}' for n, p in INPUTS if not p.exists()]
assert not missing, 'missing (run k5dfidota.ipynb first?) -- paste to Claude:\n' + '\n'.join(missing)
print(f'{len(INPUTS)} inputs found | arms {list(ARMS)} | runs / reads -> {ROOT}')

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu scikit-learn openpyxl xlrd
python -c "import av, einops, faiss, sklearn, openpyxl; print('deps OK')"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- steps 7, 8 need one'
df -h /content | tail -1

In [ ]:
import numpy as np  # noqa: E402
import torch  # noqa: E402
import yaml  # noqa: E402

from core.config import Config                       # noqa: E402
from core.data.v2_splits import load_split           # noqa: E402
from core.tools.dada_whole_eval import test_sources  # noqa: E402
from core.tools.dota_train_extract import train_clips  # noqa: E402
from core.tools.v5_stage1 import _flat               # noqa: E402

ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2')   # transformers must not import TF


def run(cmd, log=None):
    """Stream a child's output here (and to ``log``); raise with its tail on failure."""
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def stage(src, dst, ids=None):
    """Copy a Drive dir (only ``ids`` + json/npz manifests if given) or file to VM disk once (bit-identical; C2)."""
    t0 = time.time()
    if src.is_file():
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists():
            shutil.copy2(src, dst)
        return dst
    dst.mkdir(parents=True, exist_ok=True)
    names = [f.name for f in src.iterdir() if f.is_file()] if ids is None else \
        [f'{v}.npy' for v in ids] + [m.name for p in ('*.json', '*.npz') for m in src.glob(p)]
    todo = [n for n in names if not (dst / n).exists()]
    with ThreadPoolExecutor(16) as pool:
        list(pool.map(lambda n: shutil.copy2(src / n, dst / n), todo))
    print(f'  staged {src.name}: {len(todo)} files in {time.time() - t0:.0f}s')
    return dst


def publish(src, dst):
    """VM-local result -> Drive ATOMICALLY: ``<dst>.partial``, fsync, rename (C10)."""
    if src.is_dir():
        for f in [p for p in src.rglob('*') if p.is_file()]:
            publish(f, dst / f.relative_to(src))
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.partial')
    shutil.copy2(src, tmp)
    with open(tmp, 'rb+') as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, dst)


def flush_drive():
    drive.flush_and_unmount()
    drive.mount('/content/drive')


def write_ids(path, ids):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(''.join(f'{v}\n' for v in ids))
    return path


def on_drive(folder, min_bytes=1024):
    """Ids of every finished ``.npy`` in ``folder`` from ONE listing (save_array = tmp + rename)."""
    if not folder.is_dir():
        return set()
    with os.scandir(folder) as entries:
        return {e.name[:-4] for e in entries if e.name.endswith('.npy') and e.stat().st_size >= min_bytes}


def metrics_rows(path):
    return [json.loads(l) for l in path.read_text().splitlines() if l.strip()] if path.exists() else []


def ckpt_step(path):
    """The checkpoint's own global_step; 0 if absent, -1 if unreadable (never equals a real step)."""
    if not path.exists():
        return 0
    try:
        return int(torch.load(path, map_location='cpu', weights_only=False).get('global_step', 0))
    except (RuntimeError, EOFError, pickle.UnpicklingError) as exc:
        print(f'UNREADABLE {path}: {exc}')
        return -1


def is_verified(run_dir, steps):
    rec, ckpt = run_dir / 'verified.json', run_dir / 'checkpoint_last.pt'
    if rec.exists() and ckpt.exists():
        r = json.loads(rec.read_text())
        return r['step'] == steps and r['ckpt_bytes'] == ckpt.stat().st_size
    return False


def manifest(d):
    return json.loads((d / constants.V2_INPUT_MANIFEST_FILENAME).read_text())


DEFAULTS = _flat(Config().to_dict())


def config_diff(base, got):
    """Keys that differ; a key added after ``base`` trained may sit at its default."""
    out = {}
    for k in set(base) | set(got):
        if base.get(k) == got.get(k):
            continue
        if k not in base and got.get(k) == DEFAULTS.get(k):
            continue
        if k not in got and base.get(k) == DEFAULTS.get(k):
            continue
        out[k] = (base.get(k), got.get(k))
    return out

from core.data.v2_splits import lines_sha1          # noqa: E402

# Preflight on small files only (C39)
ctrl_cfg = _flat(yaml.safe_load((CTRL_RUN / 'config.yaml').read_text()))
assert all(ctrl_cfg[k] for k in ('kim.enabled', 'kim.prototypes', 'kim.edge_mask', 'kim.seam_mask')), 'CTRL_RUN is not K5dfiDoTA'
assert (ctrl_cfg['dvs.theta'], ctrl_cfg['loss.dvs_anchor_mode'], ctrl_cfg['data.is_egocentric'], ctrl_cfg['data.dataset'],
        ctrl_cfg['loss.bottomk_weight']) == (0.7, IGN, False, constants.DOTA_TRAIN_DATASET, 0.0), 'CTRL_RUN is not K5dfiDoTA'
build = json.loads(BUILD.read_text())
info, PLAN = build['construction'], build['plan']
assert (info['train_items'], info['train_abnormal'], info['train_normal'], info['clips_with_windows']) == (12937, 9386, 3551, 3019), f'not §55 DC2: {info}'
STEPS, EPOCHS = PLAN['steps'], PLAN['epochs']
assert (STEPS, EPOCHS) == (1764, 6), PLAN
assert is_verified(CTRL_RUN, STEPS) and metrics_rows(CTRL_RUN / 'metrics.jsonl')[-1]['global_step'] == STEPS, 'K5dfiDoTA: J10'
TRAIN = train_clips(constants.DOTA_TRAIN_SPLITS_DIR / constants.DOTA_TRAIN_MAP_FILENAME)
ids_all = sorted(TRAIN)
windows = json.loads((DATA / constants.WINDOWS_FILENAME).read_text())
TRAIN_SRC = sorted({w['source'] for w in windows.values()})
SHA_DOTA = lines_sha1(TRAIN_SRC)
m = manifest(BAKE)
assert (m['crn'], m['motion'], m['train_ids_sha1'], m['baked_ids']) == (OFF, ENC, SHA_DOTA, len(ids_all)), f'bake: {m}'
DADA_IDS, CAP_IDS = test_sources(T2), sorted(load_split(CAP_DEV))
NEXAR_IDS = sorted(load_split(NEXAR_SPLIT))
TAD_IDS = sorted(json.loads((TAD / constants.FRAME_LABELS_TEST_FILENAME).read_text()))
DAD_IDS = sorted(json.loads((DAD_DATA / constants.FRAME_LABELS_TEST_FILENAME).read_text()))
ZS_IDS = {'dada': DADA_IDS, 'dota': CAP_IDS, 'tad': TAD_IDS, DAD_SET: DAD_IDS}
for s, d in ZS_DST.items():
    mm = manifest(d)
    assert (mm['crn'], mm['motion'], mm['train_ids_sha1'], mm['baked_ids']) == (OFF, ENC, SHA_DOTA, len(ZS_IDS[s])), f'{s}: {mm}'
assert len(TRAIN) == 3020 and len(TRAIN_SRC) == 3019 and not set(TRAIN) & set(CAP_IDS)
print(f'preflight OK: K5dfiDoTA step {STEPS} | corpus {info} | DoTA-CAP-dev {len(CAP_IDS)}, DADA {len(DADA_IDS)}, '
      f'TAD {len(TAD_IDS)}, DAD {len(DAD_IDS)}, {NEXAR_SPLIT} {len(NEXAR_IDS)}')

## 5. Tests on the touched code (CPU) — **raises if anything fails: stop and paste the output**

In [ ]:
run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', 'core/tests/test_dota_train_corpus.py',
     'core/tests/test_kim.py', 'core/tests/test_dvs_dataset.py', 'core/tests/test_zs_compare.py'])

## 6. Stage the §55 corpus, DoTA-statistics bake and KNN (nothing is rebuilt)

In [ ]:
L_DATA  = stage(DATA, LOCAL / 'stage' / 'data')
L_BAKE  = stage(BAKE, LOCAL / 'stage' / 'bake', ids_all)
L_STATS = stage(BAKE, LOCAL / 'stage' / 'dota_stats', [])          # manifest + statistics only
L_KNN   = stage(KNN, LOCAL / 'stage' / 'knn' / KNN.name)
assert (DATA / constants.DEFS_FILENAME).read_text() == (WINDOW / constants.DEFS_FILENAME).read_text(), 'defs differ from Nexar_window'
assert len(list(L_BAKE.glob('*.npy'))) == len(ids_all), 'bake staged partially -- re-run this cell'
print(f'staged: corpus {L_DATA}, bake {len(ids_all)} clips, KNN {L_KNN.stat().st_size / 2**20:.0f} MB')

## 7. Train the three arms (§56 DA1 / DA3)

**Watch the log: past 2 s/step, stop and paste it.** Each arm = K5dfiDoTA's command + its sets in `constants.KIM_DOTA_TRAIN_ARMS`. Guards per arm: J10 at 1,764, config diff to K5dfiDoTA = exactly the keys it sets, last-epoch `mul_mil` > 0.01. An arm already verified on Drive is not retrained.

In [ ]:
BASE_SETS = ['train.stage=2', f'train.seed={SEED}', f'train.num_epochs={EPOCHS}', 'train.amp=true',
             f'data.dataset={constants.DOTA_TRAIN_DATASET}', f'data.frame_stride={constants.DOTA_TRAIN_STRIDE}',
             f'loss.mil_topk_pct={constants.DADA_ORIGIN_MIL_TOPK_PCT}', 'kip.enabled=false', f'loss.dvs_anchor_mode={IGN}',
             'data.is_egocentric=false', f'v2.motion={ENC}', f'model.motion_dim={constants.V5_GRID_DIM}',
             f'model.score_head_kernel={constants.DADA_ORIGIN_SCORE_HEAD_KERNEL}',
             'kim.enabled=true', 'kim.prototypes=true', 'kim.edge_mask=true', 'kim.seam_mask=true',
             'dvs.theta=0.7', 'dvs.theta_ego=0.85']   # = K5dfiDoTA's command (§55 notebook, step 8)


def arm_sets(row):
    """K5dfiDoTA's sets with the arm's keys replaced (a later --set would win anyway; this keeps one per key)."""
    extra = dict(s.split('=', 1) for s in ARMS[row])
    return [s for s in BASE_SETS if s.split('=', 1)[0] not in extra] + list(ARMS[row])


def last_epoch(rows, key):
    last = [r for r in rows if r['epoch'] == rows[-1]['epoch']]
    return sum(r[key] for r in last) / len(last)


def check_run(row):
    run_dir, ckpt, _ = row_paths(row)
    mrows = metrics_rows(run_dir / 'metrics.jsonl')
    assert ckpt_step(ckpt) == mrows[-1]['global_step'] == STEPS, f'{row}: J10'
    diff = config_diff(ctrl_cfg, _flat(yaml.safe_load((run_dir / 'config.yaml').read_text())))
    want = {s.split('=', 1)[0] for s in ARMS[row]}
    assert set(diff) == want, f'{row} vs {CTRL}: {diff}, expected keys {want} (§56 DA3)'
    mul = last_epoch(mrows, 'mul_mil')
    assert mul > constants.KIM_FIXED_MUL_MIL_FLOOR, f'{row}: last-epoch mul_mil {mul:.4f} -- paste to Claude'
    ref = metrics_rows(CTRL_RUN / 'metrics.jsonl')
    print(f'{row}: diff {diff}')
    for key in ('mil', 'mul_mil', 'kim_mil'):
        print(f'  last-epoch {key} {last_epoch(mrows, key):.4f} ({CTRL} {last_epoch(ref, key):.4f})')


def train_row(row):
    run_dir, ckpt, _ = row_paths(row)
    if is_verified(run_dir, STEPS):
        print(f'{row} s{SEED}: verified on Drive -- not retrained')
        check_run(row)
        return
    assert torch.cuda.is_available(), 'torch sees no GPU -- train.device=auto would silently train on CPU'
    local = LOCAL / 'runs' / row
    local.mkdir(parents=True, exist_ok=True)
    resume = ['--resume', local / 'checkpoint_last.pt'] if (local / 'checkpoint_last.pt').exists() else []
    (local / 'env.json').write_text(json.dumps({'gpu': torch.cuda.get_device_name(0), 'torch': torch.__version__, 'cuda': torch.version.cuda}))
    t0 = time.time()
    run([sys.executable, '-m', 'core.train', *[x for s in arm_sets(row) for x in ('--set', s)], '--data-dir', L_DATA,
         '--clip-dir', L_BAKE, '--output-dir', local, '--knn-cache', L_KNN, *resume], log=LOCAL / f'train_{row}.log')
    last = metrics_rows(local / 'metrics.jsonl')[-1]
    assert ckpt_step(local / 'checkpoint_last.pt') == last['global_step'] == STEPS, f'{row}: J10 locally {last["global_step"]}'
    for name in ('checkpoint_last.pt', 'metrics.jsonl', 'config.yaml', 'env.json'):
        publish(local / name, run_dir / name)
    flush_drive()
    assert ckpt_step(ckpt) == STEPS, f'{row}: Drive copy is not step {STEPS}'
    (local / 'verified.json').write_text(json.dumps({'step': STEPS, 'ckpt_bytes': ckpt.stat().st_size}))
    publish(local / 'verified.json', run_dir / 'verified.json')
    print(f'{row}: trained in {(time.time() - t0) / 60:.0f} min')
    check_run(row)


for row in ARMS:
    print(f'=== {row}: + {list(ARMS[row])}')
    train_row(row)

## 8. Reads per arm: DoTA-CAP-dev (in-domain), DADA, TAD, DAD, Nexar val — §55's protocols, on the DoTA-statistics bakes

In [ ]:
from core.tools.nexar_eval import ITEM_AUCS_JSON, READOUT_JSON, VIDEO_STATS_JSON  # noqa: E402

L_T2 = LOCAL / 'T2_meta'
L_T2.mkdir(parents=True, exist_ok=True)
shutil.copy2(T2 / constants.META_FILENAME, L_T2 / constants.META_FILENAME)
L_TAD = LOCAL / 'TAD_meta'
L_TAD.mkdir(parents=True, exist_ok=True)
for f in TAD.iterdir():                     # json / txt only -- never a frame folder
    if f.is_file() and f.suffix in ('.json', '.txt'):
        shutil.copy2(f, L_TAD / f.name)
L_DEFS = LOCAL / 'nexar_defs'
L_DEFS.mkdir(parents=True, exist_ok=True)
for f in WINDOW.glob('*.json'):             # the definitions every Nexar row read with
    shutil.copy2(f, L_DEFS / f.name)


def read_evaluate(row, name, dataset, data_dir, bake, ids, out, ckpt):
    if not (out / 'results.json').exists():
        local = LOCAL / 'eval' / row / name
        shutil.rmtree(local, ignore_errors=True)
        run([sys.executable, '-m', 'core.evaluate', '--ckpt', ckpt, '--set', f'data.dataset={dataset}',
             '--data-dir', data_dir, '--clip-dir', stage(bake, LOCAL / 'stage' / f'{name}_bake', ids),
             '--score-norm', 'auto', '--save-scores', '--output-dir', local], log=LOCAL / f'{row}_{name}.log')
        publish(local, out)
    r = json.loads((out / 'results.json').read_text())
    assert r['score_norm'] == 'none' and r['num_videos'] == len(ids), r
    print(f"{row} {name}: macro {r['auc_macro']:.4f} over {r['auc_macro_videos']} | micro {r['auc']:.4f} | AP {r['ap']:.4f}")


def read_zero_shot(row):
    _, ckpt, ev = row_paths(row)
    out = ev / 'dota' / K5
    if not (out / 'clip_aucs.json').exists():
        local = LOCAL / 'eval' / row / 'dota'
        shutil.rmtree(local, ignore_errors=True)
        run([sys.executable, '-m', 'core.tools.protocol_b_eval', '--run', f's{SEED}', ckpt,
             '--input-dir', stage(ZS_DST['dota'], LOCAL / 'stage' / 'cap', CAP_IDS), '--s1-dir', stage(DOTA_S1, LOCAL / 'stage' / 'dota_s1', CAP_IDS),
             '--metadata', DOTA / 'metadata_val.json', '--split-file', DOTA / 'val_split.txt',
             '--data-dir', stage(DOTA / 'labels_s8', LOCAL / 'stage' / 'dota_labels'), '--split', CAP_DEV,
             '--out-dir', local, '--device', 'auto'], log=LOCAL / f'{row}_dota.log')
        publish(local, out)
    print(f"{row} DoTA-CAP-dev: macro {json.loads((out / 'protocol_b_readout.json').read_text())['macro']['mean']:.4f}")
    out = ev / 'dada' / K5
    if not (out / 'clip_aucs.json').exists():
        local = LOCAL / 'eval' / row / 'dada'
        shutil.rmtree(local, ignore_errors=True)
        run([sys.executable, '-m', 'core.tools.dada_whole_eval', '--run', f's{SEED}', ckpt, '--t2-dir', L_T2,
             '--annotation', ANNOT, '--census', T2_CENSUS, '--input-dir', stage(ZS_DST['dada'], LOCAL / 'stage' / 'dada', DADA_IDS),
             '--data-dir', L_DEFS, '--out-dir', local], log=LOCAL / f'{row}_dada.log')
        publish(local, out)
    print(f"{row} DADA whole: macro {json.loads((out / 'dada_readout.json').read_text())['macro']['mean']:.4f}")
    read_evaluate(row, 'tad', TAD_NAME, L_TAD, ZS_DST['tad'], TAD_IDS, ev / 'tad' / K5 / f's{SEED}', ckpt)
    read_evaluate(row, 'dad', DAD_NAME, stage(DAD_DATA, LOCAL / 'stage' / 'dad_meta'), ZS_DST[DAD_SET], DAD_IDS,
                  ev / DAD_SET / K5 / f's{SEED}', ckpt)
    flush_drive()


def read_nexar(row):
    _, ckpt, ev = row_paths(row)
    out = ev / 'nexar' / row
    done = all((out / f).exists() for f in (READOUT_JSON, ITEM_AUCS_JSON, VIDEO_STATS_JSON)) and \
        len(list((out / 'scores').glob('*.npz'))) == len(NEXAR_IDS)
    if not done:
        L_NX = stage(NEXAR_X, LOCAL / 'stage' / 'nexar_x', NEXAR_IDS)
        L_NU = stage(NEXAR_U, LOCAL / 'stage' / 'nexar_u', NEXAR_IDS)
        local = LOCAL / 'eval' / row / 'nexar'
        shutil.rmtree(local, ignore_errors=True)
        run([sys.executable, '-m', 'core.tools.nexar_eval', '--run', f's{SEED}', ckpt, '--stats-dir', L_STATS,
             '--video-s1-dir', L_NU, '--census', CENSUS, '--clip-s1-dir', L_NX,
             '--data-dir', L_DEFS, '--split', NEXAR_SPLIT, '--save-scores', '--out-dir', local], log=LOCAL / f'{row}_nexar.log')
        publish(local, out)
        flush_drive()
    r = json.loads((out / READOUT_JSON).read_text())
    assert r['split'] == NEXAR_SPLIT and r['videos'] == len(NEXAR_IDS), (r['split'], r['videos'])
    assert r['global_steps'] == {f's{SEED}': STEPS} and Path(r['checkpoints'][f's{SEED}']) == ckpt, (r['global_steps'], r['checkpoints'])
    assert r['stats_manifest']['train_ids_sha1'] == SHA_DOTA, 'Nexar val was read with other statistics'
    print(f"{row}: crops macro {r['crops']['mean_over_placements']['macro']['mean']:.4f} | "
          f"whole macro {r['whole']['macro']['mean']:.4f} | clip AUC {r['whole']['clip_auc_max']:.4f}")


for row in ARMS:
    read_zero_shot(row)
    read_nexar(row)

## 9. Arms vs K5dfiDoTA + DAD placebo -> **paste `REPORTS/k5dfidota_arms_report.md` to Claude**

In [ ]:
from core.tools import k5_whole_report as kw     # noqa: E402

local = LOCAL / 'report'
shutil.rmtree(local, ignore_errors=True)
local.mkdir(parents=True)
EV = {CTRL: CTRL_EV, **{row: row_paths(row)[2] for row in ARMS}, NEXROW: NEX_EV}
table_args = [a for row, ev in EV.items() for s in ('dota', 'dada', 'tad', DAD_SET) for a in ('--read', s, row, ev / s / K5)]
NAME = constants.KIM_DOTA_TRAIN_ARMS_TABLE_NAME
run([sys.executable, '-m', 'core.tools.zs_arm_table', '--control', CTRL, '--seed', SEED, *table_args, '--name', NAME, '--out-dir', local])

# DAD position diagnostics for every row (§56 DA5): placebo + same-position, control K5dfiDoTA, LaGoVAD printed
dad_rows = [a for row, d in ((LAG, LAG_DAD), (CTRL, CTRL_EV / DAD_SET / K5 / f's{SEED}'),
                             *[(row, row_paths(row)[2] / DAD_SET / K5 / f's{SEED}') for row in ARMS])
            for a in ('--row', row, d)]
run([sys.executable, '-m', 'core.tools.dad_position_diag', *dad_rows, '--control', CTRL, '--out-dir', local / 'dad_position'])

lines = ['# §56 arms on DoTA train (KIM_RUN.md §56), s2024, n = 1 — printed, decides nothing; K5f stays', '',
         f"Corpus {info}; {STEPS} steps each; control {CTRL} (§55, reused). Every row is IN-DOMAIN on DoTA-CAP-dev "
         f"(position ruler 0.8545 [0.8386, 0.8695]; 566 / 569 dev clips share a YouTube id with train).", '',
         '| row | sets beyond K5dfiDoTA | GPU |', '|---|---|---|']
for row in ARMS:
    env = json.loads((row_paths(row)[0] / 'env.json').read_text())
    lines.append(f"| {row} | {', '.join(ARMS[row])} | {env['gpu']} |")
lines += ['', '## Nexar val (zero-shot for every row here; crops macro, paired per video vs K5dfiDoTA)', '',
          '| row | macro crops | macro whole | micro | AP | Δ crops vs K5dfiDoTA |', '|---|---|---|---|---|---|']
ctrl_cells = None
for row in ARMS:
    nx = kw.nexar_set({kw.K5: CTRL_EV / 'nexar' / CTRL, row: row_paths(row)[2] / 'nexar' / row}, new=row)   # the reader keys the control as K5
    c, dl = nx['rows'][row], nx['delta']
    ctrl_cells = nx['rows'][kw.K5]
    lines.append(f"| {row} | {c['macro_crops']['mean']:.4f} | {c['macro_whole']['mean']:.4f} | {c['micro']:.4f} | {c['ap']:.4f} | "
                 f"{dl['mean']:+.4f} [{dl['low']:+.4f}, {dl['high']:+.4f}] |")
lines.insert(-len(ARMS), f"| {CTRL} | {ctrl_cells['macro_crops']['mean']:.4f} | {ctrl_cells['macro_whole']['mean']:.4f} | "
                         f"{ctrl_cells['micro']:.4f} | {ctrl_cells['ap']:.4f} | — |")
lines += ['', f'## Four sets (control {CTRL}; {NEXROW} = Nexar-trained, printed)', '',
          (local / f'{constants.ZS_ARM_TABLE_STEM}_{NAME}.md').read_text(), '',
          '## DAD: placebo and same-position (post-hoc kind of read, printed for every row by §56 DA5)', '',
          (local / 'dad_position' / 'dad_position_diag.md').read_text()]
REPORT = local / f'{constants.KIM_DOTA_TRAIN_ARMS_REPORT_STEM}.md'
REPORT.write_text('\n'.join(lines) + '\n')
publish(local / f'{constants.ZS_ARM_TABLE_STEM}_{NAME}.md', REPORTS / f'{constants.ZS_ARM_TABLE_STEM}_{NAME}.md')
publish(local / f'{constants.ZS_ARM_TABLE_STEM}_{NAME}.json', REPORTS / f'{constants.ZS_ARM_TABLE_STEM}_{NAME}.json')
publish(local / 'dad_position', REPORTS / 'dad_position_arms')
publish(REPORT, REPORTS / REPORT.name)
print((REPORTS / REPORT.name).read_text())
drive.flush_and_unmount()          # C10: every pending Drive upload lands before the VM goes
print('Drive flushed -- paste REPORTS/k5dfidota_arms_report.md to Claude')